# OpenCV edge detection: USB webcam to HDMI monitor

Applies edge detection filters to the webcam picture and shows the result on an HDMI monitor,
like the EDGE ZYNQ edge detection demo video. The slide switches pick the filter while it runs.

**You need**
* a USB webcam on the USB host connector **J3**, with jumpers **J6 and J7 fitted** (host mode)
* an HDMI monitor on the HDMI connector and/or a VGA monitor on connector P1 (both show the
  same picture; connect the monitor before you run this notebook)

## 1. Load the overlay, start the HDMI output and open the webcam

In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_BGR
import os
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
import cv2
import numpy as np

base = BaseOverlay("base.bit")
hdmi_out = base.video.hdmi_out
hdmi_out.configure(VideoMode(640, 480, 24), PIXEL_BGR)
hdmi_out.start();

cam = cv2.VideoCapture(0, cv2.CAP_V4L2)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
print("Webcam open:", cam.isOpened())

def grab():
    """One webcam picture as 640x480 BGR (some webcams ignore the size request)."""
    ok, img = cam.read()
    if ok and img.shape[:2] != (480, 640):
        img = cv2.resize(img, (640, 480))
    return ok, img

## 2. The filters

In [ ]:
def laplacian(img):
    return cv2.Laplacian(img, cv2.CV_8U, ksize=3)

def canny(img):
    edges = cv2.Canny(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 100, 110)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

def sobel(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_16S, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_16S, 0, 1, ksize=3)
    mag = cv2.addWeighted(cv2.convertScaleAbs(gx), 0.5, cv2.convertScaleAbs(gy), 0.5, 0)
    return cv2.cvtColor(mag, cv2.COLOR_GRAY2BGR)

FILTERS = [("camera", lambda img: img), ("Laplacian", laplacian), ("Canny", canny), ("Sobel", sobel)]

## 3. One picture with each filter

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt
ok, frame = grab()
if not ok:
    raise RuntimeError("could not read from the webcam")
plt.figure(figsize=(12, 9))
for i, (name, f) in enumerate(FILTERS):
    plt.subplot(2, 2, i + 1)
    plt.imshow(f(frame.copy())[:, :, ::-1])
    plt.title(name)
    plt.axis("off")
plt.show()

## 4. Live edge detection on the HDMI monitor

The two left slide switches (switch 0 = SW6, switch 1 = SW4) pick the filter: 0 = camera,
1 = Laplacian, 2 = Canny, 3 = Sobel. The 7-segment display shows the frames per second.
Press **push button 0** (SW7, the left button) to stop
(it also stops after 2 minutes).

In [ ]:
import time
frames, start, shown = 0, time.time(), time.time()
while time.time() - start < 120 and not base.buttons[0].read():
    ok, frame = grab()
    if not ok:
        continue
    name, f = FILTERS[base.switches.read() & 0x3]
    out = hdmi_out.newframe()
    out[:] = f(frame)
    hdmi_out.writeframe(out)
    frames += 1
    if time.time() - shown > 1:
        base.seven_seg.show_number(round(frames / (time.time() - start), 1))
        shown = time.time()
print("{:.1f} frames per second".format(frames / (time.time() - start)))

## 5. Release the webcam and stop the HDMI output

In [ ]:
cam.release()
hdmi_out.close()
base.seven_seg.clear()